# Agno Agent — High-Level Agent Framework with Minimal Boilerplate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/85-agno-agent/agno_agent_workbook.ipynb)

**Agno** is a lightweight, high-performance agent framework that applies
convention-over-configuration to AI agents.
Where LangGraph asks you to declare `StateGraph` with nodes, edges, and a typed state dict,
Agno asks for three things: a `model`, a list of `tools`, and a `description`.

**What you'll learn:**
- How Agno reduces agent setup to 3–5 lines
- How to define tools as plain Python functions
- How to inspect `RunResponse` for tool calls and message history
- How Agno `Team` compares to LangGraph supervisor patterns

## Workshop Roadmap

| # | Section | Exercise |
|---|---------|----------|
| 1 | Agno's Design Philosophy | Compare setup complexity |
| 2 | Creating and Running an Agent | Query with tool calls |
| 3 | RunResponse Anatomy | Inspect messages + tools |
| 4 | Multi-Agent Teams | Coordinate + route modes |
| 5 | When to Choose Agno vs LangGraph | Decision guide |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "agno", "openai", "python-dotenv"],
        check=True,
    )
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print("API key loaded." if key else "WARNING: OPENAI_API_KEY not set.")

## Part 1 — Agno's Design Philosophy

Agno applies **convention over configuration** to AI agents.
The framework infers tool schemas from Python docstrings, manages the reasoning loop internally,
and returns a clean `RunResponse` object.

**Lines of code comparison for a simple tool-calling agent:**

| Step | LangGraph | Agno |
|------|-----------|------|
| State definition | `TypedDict` + fields | — |
| Graph construction | `StateGraph(State)` | — |
| Node functions | 1+ node defs | — |
| Tool dispatch | Manual in node | — |
| Edge wiring | `add_edge` / `add_conditional_edges` | — |
| Compile + run | `compile()` + `invoke()` | — |
| **Agent declaration** | — | `Agent(model, tools, description)` |
| **Run** | — | `agent.run(query)` |

> **Agno optimizes for time-to-prototype; LangGraph optimizes for correctness and debuggability.**

| Dimension | Agno | LangGraph |
|-----------|------|----------|
| Setup complexity | ~5 lines | ~30–50 lines |
| State management | Internal | Explicit typed state |
| Multi-agent | `Team` with `coordinate`/`route` | Subgraphs + supervisor |
| Debugging | `RunResponse.messages` | Full node-level state trace |
| Production suitability | Rapid prototyping | Full control, audit trails |

In [ ]:
# Agno reads plain Python docstrings for tool schema — no decorators needed.

KNOWLEDGE: dict = {
    "python":    "Python is a high-level, dynamically typed language.",
    "agno":      "Agno is a lightweight, high-performance agent framework.",
    "langgraph": "LangGraph builds stateful agent graphs with explicit state.",
}


def search_knowledge(query: str) -> str:
    '''Search the knowledge base for a topic.

    Args:
        query: The topic name to search (e.g. 'python', 'agno').

    Returns:
        A description of the topic, or a not-found message.
    '''
    result = KNOWLEDGE.get(query.lower().strip())
    return result if result else f"No entry found for '{query}'."


def add_knowledge(topic: str, description: str) -> str:
    '''Add a new topic to the knowledge base.

    Args:
        topic: The topic name to store.
        description: A short description of the topic.

    Returns:
        A confirmation message.
    '''
    KNOWLEDGE[topic.lower().strip()] = description
    return f"Stored: '{topic}'"


print("Tools defined. Agno will inspect their docstrings for the tool schema.")
print(f"search_knowledge doc: {search_knowledge.__doc__.strip().splitlines()[0]}")

## Part 2 — Creating and Running an Agent

`Agent(model, tools, description)` is the complete setup.

Key constructor arguments:
- `model` — an `OpenAIChat`, `Gemini`, or other Agno model wrapper
- `tools` — list of plain Python functions; Agno extracts schema from docstrings
- `description` — the system prompt equivalent
- `show_tool_calls=True` — prints each tool invocation to stdout (useful for learning)
- `markdown=False` — returns plain text instead of Markdown-formatted output

`agent.run(query)` returns a `RunResponse`.
`response.content` is the final text answer.

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

agent = Agent(
    model=OpenAIChat(id="gpt-5.4-nano"),
    description="You are a helpful assistant with a technology knowledge base.",
    tools=[search_knowledge, add_knowledge],
    show_tool_calls=True,
    markdown=False,
)

queries = [
    "What is Agno?",
    "What is LangGraph?",
    "Add this: 'redis' is 'an in-memory key-value store'.",
    "Look up redis.",
]

for query in queries:
    print(f"\nQ: {query}")
    response = agent.run(query)
    print(f"A: {response.content}")

## Part 3 — RunResponse Anatomy

A `RunResponse` carries more than just the final answer:

| Attribute | Contents |
|-----------|----------|
| `.content` | Final text response from the model |
| `.messages` | Full message history (system, user, assistant, tool) |
| `.tools` | Tool calls made during this run |

Inspecting `.messages` lets you trace the full conversation arc:
what the system prompt was, what the user sent, what tools were called, and what the model replied.

This is Agno's equivalent of LangGraph's node-level state inspection.

In [ ]:
response = agent.run("What do you know about Python and LangGraph?")

print(f"Content type: {type(response.content)}")
print(f"Content: {response.content}")
print()

# Messages shows the full conversation arc.
if hasattr(response, 'messages') and response.messages:
    print(f"Messages in response: {len(response.messages)}")
    for msg in response.messages:
        role = getattr(msg, 'role', 'unknown')
        content = getattr(msg, 'content', '')
        if content and isinstance(content, str):
            preview = content[:80].replace('\n', ' ')
            print(f"  [{role}] {preview}...")

## Part 4 — Multi-Agent Teams

Agno `Team` lets multiple agents collaborate under a team leader.

Two modes:
- `mode="coordinate"` — the team leader decomposes the task and delegates sub-tasks to members
- `mode="route"` — the team leader picks exactly one member to handle the full request

**Compare with LangGraph supervisor (example 80):**

| Aspect | LangGraph supervisor | Agno Team |
|--------|---------------------|----------|
| Routing declaration | Dynamic — conditional edges at runtime | Static — declared at construction |
| Sub-agent communication | Via shared state dict | Via Team leader orchestration |
| Visibility | Full state trace | `RunResponse.messages` |
| Flexibility | High | Moderate |

In [ ]:
from agno.team import Team

researcher = Agent(
    model=OpenAIChat(id="gpt-5.4-nano"),
    name="Researcher",
    description="You research technology topics using the knowledge base.",
    tools=[search_knowledge],
)

writer = Agent(
    model=OpenAIChat(id="gpt-5.4-nano"),
    name="Writer",
    description="You write clear, concise summaries of research findings.",
)

team = Team(
    name="Research Team",
    mode="coordinate",
    model=OpenAIChat(id="gpt-5.4-nano"),
    members=[researcher, writer],
    show_tool_calls=True,
    markdown=False,
)

task = "Research what LangGraph is and write a two-sentence beginner summary."
print(f"Task: {task}\n")
response = team.run(task)
print(f"Result:\n{response.content}")

## Part 5 — When to Choose Agno vs LangGraph

**Choose Agno when:**
- You need a working prototype in under an hour
- The agent is a simple tool-calling loop with no complex branching
- Minimal boilerplate is more important than full control
- You want a `Team` without writing supervisor graph logic

**Choose LangGraph when:**
- You need complex conditional branching (retry loops, escalation paths)
- Human-in-the-loop interrupts are required
- Persistent memory across sessions matters (`MemorySaver`, `SqliteSaver`)
- You need a full audit trail of every state transition
- Production systems where debuggability is non-negotiable

**Both support multi-agent.** Agno's `Team` is faster to set up.
LangGraph's supervisor gives you more granular control over routing logic.

In [ ]:
# Side-by-side: same task, different API surface areas.

print("=== Agno approach ===")
simple_agent = Agent(
    model=OpenAIChat(id="gpt-5.4-nano"),
    tools=[search_knowledge],
    description="Answer questions about technology.",
    markdown=False,
)
r = simple_agent.run("What is Python?")
print(r.content)

print()
print("=== LangGraph equivalent would need: ===")
print("  1. TypedDict state definition")
print("  2. StateGraph construction")
print("  3. Node functions")
print("  4. Edge wiring")
print("  5. compile()")
print("  6. app.invoke()")
print()
print("Trade-off: Agno = fast prototyping; LangGraph = full control + debuggability")

## Exercises

**Exercise 1 — Calculator agent**

Create an agent with a `calculator` tool that evaluates basic arithmetic expressions.
Test it with at least 3 math queries including one that uses exponentiation.

**Exercise 2 — Three-member team**

Create a three-member `Team`: Researcher, Analyst, Writer.
- Researcher: searches the knowledge base
- Analyst: identifies patterns and implications in the findings
- Writer: produces a polished final summary

Run the team on a complex question like: "Compare LangGraph and Agno for building production AI agents."

In [ ]:
# Exercise 1 — Calculator tool

def calculator(expression: str) -> str:
    '''Evaluate a basic mathematical expression safely.

    Args:
        expression: A math expression like "2 + 2" or "10 * 5".

    Returns:
        The result as a string.
    '''
    try:
        # eval is constrained to arithmetic — no builtins, no globals.
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


calc_agent = Agent(
    model=OpenAIChat(id="gpt-5.4-nano"),
    tools=[calculator],
    description="You are a helpful math assistant.",
    show_tool_calls=True,
    markdown=False,
)

math_queries = [
    "What is 15 * 7?",
    "Calculate 100 / 4 + 12",
    "What is 2 ** 10?",
]

for q in math_queries:
    print(f"Q: {q}")
    print(f"A: {calc_agent.run(q).content}\n")

---

**Next:** [86 — Mixture of Agents](../86-mixture-of-agents/mixture_of_agents_workbook.ipynb) — LangGraph `Send()` fan-out for parallel proposers.